# Ground state of the 2D Fermi-Hubbard model benchmark

This notebook benchmarks quantum phase estimation (QPE) for the ground state of the
**Fermi-Hubbard model on a two-dimensional square lattice**, and estimates the fault-tolerant
resources required to run it across a sweep of lattice sizes.

$$
H = -T\sum_{\langle i,j\rangle,\;\sigma\in\{\uparrow,\downarrow\}} \left(c^{\dagger}_{i\sigma}c_{j\sigma} + \text{h.c.}\right)
  \; + \; U\sum_{i} n_{i\uparrow}\,n_{i\downarrow},
\qquad n_{i\sigma}=c^{\dagger}_{i\sigma}c_{i\sigma}
$$

Here $c^{\dagger}_{i\sigma}$ and $c_{j\sigma}$ are the fermionic creation and annihilation operators on lattice sites $i,j$, and $n_{i\sigma}$ is the on-site occupation with spin $\sigma$.

**Benchmark specification**

| Quantity | Value |
|---|---|
| Lattice | 2D square, periodic in both directions, $N=L^{2}$ sites |
| Parameters | $U/T = 8$; filling $n = 0.875$ electrons per site $ |
| Target | Ground-state energy to $0.0051\,T$ per site |
| Hardware model | Majorana qubits, measurement error rate $10^{-5}$ |

**References**

- Kivlichan, Ian D., et al. "Improved fault-tolerant quantum simulation of condensed-phase correlated electrons via Trotterization." *Quantum* **4** (2020): 296. [arXiv:1902.10673](https://arxiv.org/abs/1902.10673)
- Campbell, Earl T. "Early fault-tolerant simulations of the Hubbard model." *Quantum Science & Technology* **7**.1 (2022): 015007. [arXiv:2012.09238v4](https://arxiv.org/abs/2012.09238v4)
- Bärtschi, Andreas, et al. "Potential applications of quantum computing at Los Alamos National Laboratory." (2024), Chapter 4 ("High-temperature superconductivity and exotic properties of Fermi-Hubbard models"), Application 1 ("Zero-Temperature Superconductivity"). [arXiv:2406.06625](https://arxiv.org/abs/2406.06625)

**Requirements**

```bash
pip install 'qdk-chemistry[jupyter,qre]'
```

In [ ]:
from qdk_chemistry.algorithms import create
from qdk_chemistry.data import (
    AlgorithmRef,
    Circuit,
    LatticeGraph,
    MajoranaMapping,
)
from qdk_chemistry.data.circuit import QsharpFactoryData
from qdk_chemistry.utils import Logger
from qdk_chemistry.algorithms.hamiltonian_unitary_builder.time_evolution.plaquette_trotter import (
    PlaquetteTrotter,
)
from qdk_chemistry.utils.model_hamiltonians import create_hubbard_hamiltonian
from qdk_chemistry.utils.qsharp import (
    QSHARP_UTILS,
    create_qsharp_context,
    use_qsharp_context,
)

Logger.set_global_level(Logger.LogLevel.off)

HOPPING_T = 1.0                                          # T > 0, energies are quoted in units of T
U_OVER_T = 8.0                                          # U = 8t, strong coupling: LANL Fermi-Hubbard chapter (arXiv:2406.06625 Ch. 4); also Campbell Table II (arXiv:2012.09238v4)
COULOMB_U = U_OVER_T * HOPPING_T
FILLING = 0.875                                          # electrons per SITE (Kivlichan arXiv:1902.10673 Sec. 3.2; half-filling is n=1); round(FILLING * N) electrons total
TARGET_PRECISION_PER_SITE = 0.0051   # per-site ground-state ENERGY target (units of T) chosen for this benchmark; not from LANL App. 1, which specifies accuracy on the order parameter (1e-5)
# The sparse on-demand Majorana mapper now supports 131072 qubits, enough for L <= 256
# at 2*L^2. Memory binds earlier: measured Hamiltonian-build peak RSS grows like ~L^3.7,
# from 2.0 GB at L=60 to 24.0 GB at L=120, putting L=200 near 220 GB by extrapolation.
# Run the full sweep only on a node with sufficient RAM.
MAPPER_QUBIT_LIMIT = 131_072
MAX_SWEEP_SIZE = 200
assert 2 * MAX_SWEEP_SIZE**2 <= MAPPER_QUBIT_LIMIT
BENCHMARK_LATTICE_SIZES = [
    size for size in list(range(4, 11, 2)) + list(range(20, 201, 10)) if size <= MAX_SWEEP_SIZE
]
EXAMPLE_NUM_SITES = 50

# QDK interpreters are thread-affine, so this context belongs to the notebook execution thread.
QSHARP_CONTEXT = create_qsharp_context()

def target_precision(size: int) -> float:
    """Absolute ground-state energy accuracy required of an L x L lattice."""
    return TARGET_PRECISION_PER_SITE * size * size

print(f"Hubbard model: U/T = {U_OVER_T:g}  (T = {HOPPING_T:g}, U = {COULOMB_U:g}) Filling n = {FILLING:g} electrons/site")
print(f"Benchmark lattice sizes L = {BENCHMARK_LATTICE_SIZES}")

## The lattice and the qubit Hamiltonian

`LatticeGraph.square` builds the $L\times L$ lattice and `create_hubbard_hamiltonian` turns it into a
fermionic Hamiltonian. The Jordan-Wigner mapping then produces the qubit Hamiltonian on $2N$ qubits
(one per spin-orbital).

In [ ]:
def qubit_operator(size: int):
    """Jordan-Wigner qubit Hamiltonian of the periodic size x size Hubbard lattice."""
    num_sites = size * size
    lattice = LatticeGraph.square(size, size, periodic_x=True, periodic_y=True)
    hamiltonian = create_hubbard_hamiltonian(
        lattice, epsilon=0.0, t=HOPPING_T, U=COULOMB_U
    )
    operator = create("qubit_mapper").run(
        hamiltonian, mapping=MajoranaMapping.jordan_wigner(2 * num_sites)
    )
    return operator

def num_electrons(size: int) -> int:
    """Electron count nearest to the requested per-site filling: round(n * N)."""
    return round(FILLING * size * size)

example = qubit_operator(EXAMPLE_NUM_SITES)
print(f"{EXAMPLE_NUM_SITES}x{EXAMPLE_NUM_SITES} lattice: {example.num_qubits} qubits, {len(example.pauli_strings)} Pauli terms, "
      f"{num_electrons(EXAMPLE_NUM_SITES)} electrons, lambda = {example.schatten_norm:g}")

<a id="sizing"></a>

## Sizing the QPE and Trotter parameters

For the target precision $\epsilon(L)$, the target precision is split between the two independent error sources,
$\epsilon = \epsilon_{\mathrm{QPE}} + \epsilon_{\mathrm{Trotter}}$.

**QPE Phase register width.**  Every eigenvalue of $H$ lies in $[-\lambda, \lambda]$, so

$$
t_0 = \frac{\pi}{\lambda} \\quad \lambda = \sum_j |c_j|,
$$

maps the spectrum of $H t_0$ into $[-\pi, \pi]$ and the QPE phase into $[-\tfrac{1}{2}, \tfrac{1}{2})$, using the
whole phase register.

An $m$-bit register resolves the phase to $2^{-m}$, hence the energy to
$2\lambda / 2^{m}$. Resolving $\epsilon_{\mathrm{QPE}}$ therefore costs
$\lceil \log_2 (2\lambda/\epsilon_{\mathrm{QPE}}) \rceil$ bits, and reading those leading bits correctly with
probability $1 - \delta$ costs $\lceil \log_2 (2 + 1/2\delta) \rceil$ additional guard bits:

$$
m = \left\lceil \log_2 \frac{2\lambda}{\epsilon_{\mathrm{QPE}}} \right\rceil
  + \left\lceil \log_2 \left( 2 + \frac{1}{2\delta} \right) \right\rceil .
$$

**Trotter steps.** The error that matters is on the *energy*, not on the unitary. A
second-order step of duration $s$ realizes $e^{-i H_{\mathrm{eff}} s}$ for an effective
Hamiltonian with $\lVert H_{\mathrm{eff}} - H \rVert \le W s^{2}$ (Campbell's Eq. (F2) (arXiv:2012.09238v4, App. F);
Kivlichan Eq. (8)). Repeating it is $U_{\mathrm{TS}}(s)^{r} = e^{-i H_{\mathrm{eff}} s r}$ with the
*same* $H_{\mathrm{eff}}$, so QPE reads an eigenvalue of that one fixed operator and the energy error
stays $W s^{2}$ regardless of $r$ -- it does not accumulate. Imposing $W s^{2} \le \epsilon_{\mathrm{Trotter}}$
with $t_0 = r\,s$ gives

$$
r = \left\lceil t_0 \sqrt{\frac{W}{\epsilon_{\mathrm{Trotter}}}} \right\rceil .
$$

The constant $W$ is Campbell's $W_{\mathrm{PLAQ}}$ (arXiv:2012.09238v4, Eq. (20), Sec. III;
App. D Eqs. (D6)--(D10)), derived for exactly this
two-section splitting rather than for a generic term-by-term product formula. A
term-by-term bound would not credit the fact that the plaquettes within a section are
vertex-disjoint and therefore commute, so it would demand many more steps than the
circuit actually needs.


**Allocating the budget.** 
base unitary as $U = S_2(\Delta t)^{r}$ over total time $t_0 = r\,\Delta t$, applied $2^{m}$ times by the
ladder. The step's energy error is $\epsilon_{\mathrm{Trotter}} = W \Delta t^{2}$,
while the readout resolution is $\epsilon_{\mathrm{QPE}} = 2\pi/(t_0 2^{m})$. The total cost is

$$
2^{m} \cdot r \;=\; \frac{2\pi}{t_0\,\epsilon_{\mathrm{QPE}}}\cdot\frac{t_0}{\Delta t}
\;=\; \frac{2\pi}{\Delta t \; \epsilon_{\mathrm{QPE}}},
$$

which is independent of $t_0$ and $r$ *separately* -- only the step time matters. Substituting
$\Delta t = (\epsilon_{\mathrm{Trotter}}/W)^{1/2}$ and maximizing
$\epsilon_{\mathrm{Trotter}}^{1/2}\,(\epsilon - \epsilon_{\mathrm{Trotter}})$ gives

$$
\epsilon_{\mathrm{Trotter}} = \tfrac{1}{3}\epsilon, \qquad \epsilon_{\mathrm{QPE}} = \tfrac{2}{3}\epsilon ,
$$

which is stated immediately after Campbell's Eq. (F6) (arXiv:2012.09238v4, App. F). 

That optimum assumes $m$ is continuous. Because $m$ is an integer, the resolution $2\lambda/2^{m}$ jumps by
factors of two, and a fixed $1/3$ split usually leaves part of the budget stranded. The planner below instead
enumerates $m$, hands **every** unit of leftover budget to the Trotter term (the largest $\Delta t$, hence the
smallest $r$), and keeps the $m$ that minimizes $2^{m} r$. Handing the leftover budget to the Trotter term lets a smaller $m$ still meet the target, which saves a phase bit and halves the $2^{m}$ ladder repetitions.

**The Trotter term is far from binding here.** Because $t_0 = \pi/\lambda$ is short (the 1-norm $\lambda \propto N$
grows with the lattice), a single full-time step already sits well under the Trotter budget. Measured on this
branch, the energy error of one step, $W t_0^{2}$, runs from $\approx 3.5\times10^{-2}$ at $L = 6$ (about
$2\times$ under budget) to $\approx 3.1\times10^{-5}$ at $L = 200$ (about $10^{6}\times$ under budget), so the
planner returns $r = 1$ for every executed lattice $L \ge 6$. The lone exception is the degenerate $L = 4$
tile: its two plaquette sections commute exactly (Campbell Table III, arXiv:2012.09238v4) so the true error is zero, yet the
extensive $W_{\mathrm{PLAQ}}$ bound is nonzero and rounds $r$ up to 2. The step count is 1 because the
$\lambda$-normalized step is short, not because the bound is loose.

In [ ]:
import math
from dataclasses import dataclass

import numpy as np
import pandas as pd

TROTTER_ORDER = 2                    # Suzuki-Trotter product-formula order
QPE_FAILURE_PROBABILITY = 0.1        # 1 - confidence that the readout meets the target precision
WEIGHT_THRESHOLD = 1e-12
MAX_RESOLUTION_BITS = 64

GUARD_BITS = math.ceil(math.log2(2 + 1 / (2 * QPE_FAILURE_PROBABILITY)))

@dataclass(frozen=True)
class QpeParameters:
    """Algorithm parameters derived from a target precision."""

    one_norm: float        # lambda
    evolution_time: float  # t_0
    num_bits: int          # m, including guard bits
    num_divisions: int     # r
    error_constant: float  # Campbell's W_PLAQ (arXiv:2012.09238v4, Eq. (20) and App. D)
    trotter_budget: float  # epsilon_Trotter actually allocated

def plan_qpe(one_norm: float, precision: float, error_constant: float) -> QpeParameters:
    """Choose (m, r) minimizing the ladder cost 2**m * r for a target precision.

    Implements Campbell's Appendix F optimization (arXiv:2012.09238v4) with m constrained to integers: for
    every candidate resolution the leftover budget goes entirely to the Trotter
    term, which maximizes the step time and therefore minimizes r.
    """
    evolution_time = math.pi / one_norm          # H*t_0 spectrum fits in [-pi, pi]; the endpoints +/-lambda alias, but the ground state sits strictly inside

    best = None
    for resolution_bits in range(1, MAX_RESOLUTION_BITS):
        readout_budget = 2 * one_norm / 2**resolution_bits    # = 2*pi/(t_0 * 2**m)
        if readout_budget >= precision:
            continue                                          # nothing left for Trotter
        trotter_budget = precision - readout_budget
        # Campbell's Eq. (F2) (arXiv:2012.09238v4, App. F) / Kivlichan Eq. (8): a second-order step of
        # duration s has ENERGY error W*s^2 (not the unitary-norm bound W*s^3). It
        # does not accumulate: the r repetitions realize e^{-i H_eff s r} for one
        # fixed H_eff whose distance from H is W*s^2, and QPE reads an eigenvalue of
        # that H_eff. So the largest step meeting the budget is s = sqrt(budget / W).
        step_time = math.sqrt(trotter_budget / error_constant)
        divisions = max(1, math.ceil(evolution_time / step_time))
        cost = (2**resolution_bits) * divisions
        if best is None or cost < best[0]:
            best = (cost, resolution_bits, divisions, trotter_budget)

    if best is None:
        raise ValueError(f"no resolution meets precision {precision:g} for lambda {one_norm:g}")

    _, resolution_bits, divisions, trotter_budget = best
    return QpeParameters(
        one_norm=one_norm,
        evolution_time=evolution_time,
        num_bits=resolution_bits + GUARD_BITS,
        num_divisions=divisions,
        error_constant=error_constant,
        trotter_budget=trotter_budget,
    )

def qpe_parameters(operator, precision: float, lattice_size: int) -> QpeParameters:
    """Derive the QPE register width and Trotter step count for a target precision.

    The error constant is Campbell's W_PLAQ (arXiv:2012.09238v4, Eq. (20) and App. D), which is derived for exactly the
    two-section plaquette splitting used here and reproduces his Table I. A generic
    term-by-term commutator bound would not know that each section is internally
    exact, and would be far more conservative.
    """
    constant = PlaquetteTrotter._plaquette_error_constant(lattice_size, lattice_size, HOPPING_T, COULOMB_U)
    return plan_qpe(operator.schatten_norm, precision, constant)

parameters = qpe_parameters(example, target_precision(EXAMPLE_NUM_SITES), EXAMPLE_NUM_SITES)
print(f"{EXAMPLE_NUM_SITES}x{EXAMPLE_NUM_SITES} lattice, epsilon = {target_precision(EXAMPLE_NUM_SITES):g}: {parameters}")

### Where the Trotter step count comes from

The step count $r$ is sized by Campbell's error constant (arXiv:2012.09238v4, Eq. (20), Sec. III; App. D Eqs. (D6)--(D10)) for exactly this splitting,

$$
W_{\mathrm{PLAQ}} \le W_{\mathrm{SO2}} + \tfrac{3}{24}\,\lVert [[R_p,R_g],R_g] \rVert_1 ,
\qquad
W_{\mathrm{SO2}} \le \tfrac{u\tau^{2}}{6}L^{2}(\sqrt5+8) + \tfrac{u^{2}}{24}\lVert R \rVert_1 ,
$$

evaluated by `PlaquetteTrotter._plaquette_error_constant`. That method reproduces Campbell's Table I (arXiv:2012.09238v4)
to within its two significant figures, and the library's test suite pins it there, so
the constant used here is the published one rather than a fitted stand-in.

Both trace norms are extensive, so they are computed exactly while that is affordable
and from their per-site limits above 1600 sites. The exact route is $O(N^3)$ in the
site count -- under two seconds at $L=40$, but hours and gigabytes by $L=180$ -- which
is why the cutoff exists. The 4×4 lattice is a useful sanity check: its two sections
commute, so the commutator norm is exactly zero, matching Campbell's Table III (arXiv:2012.09238v4).


## Reference state and QPE circuit

The benchmark calls for a Hartree-Fock (Fermi-sea) reference. In the site basis that the Jordan-Wigner
mapping works in, the Fermi sea is a *momentum*-space determinant, so preparing it exactly needs a
Givens-rotation network of $O(N^{2})$ two-qubit gates. We use the site-basis occupation-number determinant
with the same particle number and $S_z = 0$ instead, which is a single layer of $X$ gates. The two choices
differ in *overlap* with the ground state -- which sets how many times QPE has to be repeated -- but barely
in *cost*: $O(N^{2})$ gates against the $\sim 10^{7}$ rotations inside the QPE ladder.

The `qdk_iterative` builder then assembles a single iterative-QPE round: one controlled power
$C\text{-}U^{2^{k}}$ (here the top rung, `IQPE_ITERATION = 0`) rather than the whole $m$-rung ladder and
its inverse QFT. That controlled power is compiled by the `cswap_pauli_sequence` mapper into a Q#
operation, so the object handed to the resource estimator in the next section is the actual circuit.

The `cswap_pauli_sequence` mapper is cheaper than per-gate control (`pauli_sequence`), and it is available
*because* the plaquette product formula is vacuum-preserving. Rather than controlling every rotation, it
allocates a $|0\ldots0\rangle$ vacuum register, controlled-SWAPs the system into it, runs the
**uncontrolled** evolution, and swaps back; the $|1\rangle$ control branch accumulates the eigenphase
exactly as a $C\text{-}U$ would. With the control off the evolution runs on the true vacuum and contributes
only $U|0\ldots0\rangle = e^{i\varphi_0}|0\ldots0\rangle$ -- a global phase that `_vacuum_phase` computes
classically from the diagonal terms and cancels with an $R_1$ on the control. This holds only while the
ordered product formula keeps $|0\ldots0\rangle$ an eigenstate: the plaquette ordering does (its
$XX$/$YY$ hopping partners stay adjacent, so each pair exponentiates as one commuting block, and the
interaction terms are diagonal), and the mapper validates this and raises `ValueError` otherwise -- so it
fails loudly, not silently. `pauli_sequence` carries no such precondition.

Measured here in this repo's container (full install, `Adaptive_RIF` profile) at $L = 4$, $U/T = 8$,
`time=0.05`, `num_divisions=1` -- the single controlled power that reaches the estimator:

| Controlled mapper | qubits | T | rotations | rotation depth | Toffoli | measurements |
|---|---|---|---|---|---|---|
| `pauli_sequence` | 70 | 288 | 168 | 122 | 92 | 92 |
| `cswap_pauli_sequence` | 102 | 288 | 70 | 26 | 156 | 92 |

Pricing a synthesized rotation at $\sim 50$ T and a Toffoli at 4 T (rough figures, for orientation only), that
is 9056 vs 4412 T-equivalent: the CSWAP sandwich is about 51% cheaper *including* its extra Toffolis, and its
rotation depth is roughly $4.7\times$ shallower (26 vs 122). It pays for this in qubits -- with rotation
batching, both the vacuum register and the Hamming-weight ancillas scale with the system, so already at
$L = 4$ it uses *more* qubits than direct control (102 vs 70), and the gap widens with $L$.

## How this compares to the published plaquette costings

The plaquette decomposition used here follows Campbell (arXiv:2012.09238v4, Quantum
Sci. Technol. **7** 015007), and Bay-Smidt *et al.* (arXiv:2501.10314) apply the same
C4 construction to other lattices. The two papers agree on the per-plaquette
structure, and this implementation reproduces it: **two** non-zero eigenvalues of the
four-cycle hopping matrix, hence **two** arbitrary-angle rotations per plaquette (Campbell's
App. E, Eqs. (E11)--(E14), arXiv:2012.09238v4).

The gate counts do **not** match exactly, and it is worth being precise about why.
Campbell's Table I (arXiv:2012.09238v4) gives, per second-order Trotter step on an $L \times L$ lattice,

$$N_T = 12 L^2, \qquad N_R = 4 L^2,$$

so 192 T and 64 rotations at $L = 4$. Measured here (uncontrolled step, `Adaptive_RIF`, $L = 4$, $U/T = 8$,
`time=0.05`, `num_divisions=1`), this implementation emits **288 T** and **70** arbitrary-angle rotations,
plus **92 Toffoli and 92 measurements** from Hamming-weight phasing. Two structural choices bear on the T count:

| Difference | Campbell | Here | Effect on T |
|---|---|---|---|
| Conjugating network | 4 F-gates, 8 T per plaquette | 3 real Givens, 12 T per plaquette | $\times 1.5$ |
| Hopping half-steps | Eq. (D2) (arXiv:2012.09238v4, App. D) ordering | same ordering (was unmerged) | none now (was $\times 4/3$) |

The measured T count is now exactly $1.5 \times 192 = 288$: only the conjugating-network factor remains. An
earlier version of this notebook measured 384 T ($= 2 \times 192$); the extra $4/3$ came from an unmerged
hopping layer and vanished when the step adopted Campbell's own Eq. (D2) (arXiv:2012.09238v4, App. D) ordering
($I/2 \to P/2 \to G \to P/2 \to I/2$). The F-gate
gap is structural rather than an oversight: Campbell's network (arXiv:2012.09238v4, App. E, Eqs. (E11)--(E14)) absorbs one F-gate
pair because it acts only on modes the phase layer leaves alone, and a search over
orderings of three fixed $\pi/4$ *real* Givens rotations finds no arrangement with
that property. Closing it needs the complex fermionic-Fourier F gate, which this
implementation deliberately avoids — a real network needs no phase gates and, because
it carries the Jordan-Wigner string inside each rotation, no fermionic swap network
either. Both papers explicitly *neglect* fSWAP cost as Clifford, so their counts
assume away something this implementation genuinely does not need.

Two further caveats on comparability:

* **The step count $r$ below is sized by Campbell's $W_{\mathrm{PLAQ}}$ (arXiv:2012.09238v4, Eq. (20), Sec. III; App. D Eqs. (D6)--(D10)).** He bounds
  $W_{\mathrm{PLAQ}} \le W_{\mathrm{SO2}} + \tfrac{3}{24}\lVert[[R_p,R_g],R_g]\rVert_1$
  for exactly this two-section splitting, and `PlaquetteTrotter._plaquette_error_constant` evaluates it,
  reproducing his Table I/III. This is *not* a generic term-by-term commutator bound,
  which would not know that each section is internally exact and would be far more
  conservative (see the `trotter_steps_naive` note below).
* **Campbell (arXiv:2012.09238v4, App. E) never addresses controlled evolution.** His per-plaquette cost is for the
  uncontrolled operator. Phase estimation applies the controlled one, where a fixed
  angle costs the same as an arbitrary one, so the conjugating network here is emitted
  uncontrolled via $C(VDV^{\dagger}) = V\,C(D)\,V^{\dagger}$. That identity appears in
  neither paper.

On-site interaction: the equal-angle interaction terms are phased together through a Hamming-weight register
rather than emitted as individual rotations -- that is where the 92 Toffoli and 92 measurements come from, and
why only 70 arbitrary-angle rotations remain; Campbell (arXiv:2012.09238v4, App. E, Thm. 2) likewise
Hamming-weight-phases the interaction (his Table II budgets $L^2/2$ ancillas). The equivalent unbatched
representation is 288 T / 144 rotations / 0 Toffoli. Still not implemented here: the complex F-gate network and
the chemical-shift form of the interaction. Bay-Smidt *et al.* report no square-lattice resource numbers at all,
so there is nothing to compare against there beyond the shared per-tile structure.


In [ ]:
IQPE_ITERATION = 0                   # iteration 0 carries the largest power, 2**(m-1)

def reference_state_prep(num_sites: int, electrons: int) -> Circuit:
    """Occupation-number determinant, one X gate per occupied spin-orbital.

    Electrons are split as evenly as possible between the spin-up block
    (qubits 0..N-1) and the spin-down block (qubits N..2N-1).
    """
    num_up = (electrons + 1) // 2
    num_down = electrons // 2
    occupations = (
        [1] * num_up + [0] * (num_sites - num_up)
        + [1] * num_down + [0] * (num_sites - num_down)
    )
    with use_qsharp_context(QSHARP_CONTEXT):
        state_preparation = QSHARP_UTILS.StatePreparation
        params = state_preparation.SingleReferenceParams(bitStrings=occupations, numQubits=2 * num_sites)
        return Circuit(
            qsharp_factory=QsharpFactoryData(
                program=state_preparation.MakeSingleReferenceStateCircuit, parameter=vars(params)
            ),
            qsharp_op=state_preparation.MakePrepareSingleReferenceStateOp(params),
            encoding="jordan-wigner",
        )

def qpe_circuit(
    operator, parameters: QpeParameters, initial_state: Circuit, lattice_size: int
) -> Circuit:
    """Single IQPE iteration for `operator`, Trotterized according to `parameters`.

    `num_iteration` selects one round instead of the whole ladder, so exactly one
    controlled unitary is compiled rather than `num_bits` of them.
    """
    with use_qsharp_context(QSHARP_CONTEXT):
        builder = create(
            "qpe_circuit_builder",
            "qdk_iterative",
            unitary_builder=AlgorithmRef(
                "hamiltonian_unitary_builder",
                "plaquette",
                order=TROTTER_ORDER,
                time=parameters.evolution_time,
                num_divisions=parameters.num_divisions,
                lattice_width=lattice_size,
                lattice_height=lattice_size,
            ),
            controlled_circuit_mapper=AlgorithmRef("controlled_circuit_mapper", "pauli_sequence"),
            num_bits=parameters.num_bits,
            num_iteration=IQPE_ITERATION,
        )
        return builder.run(initial_state, operator)[0]

circuit = qpe_circuit(
    example,
    parameters,
    reference_state_prep(EXAMPLE_NUM_SITES * EXAMPLE_NUM_SITES, num_electrons(EXAMPLE_NUM_SITES)),
    EXAMPLE_NUM_SITES,
)

## Physical resource estimation

The Q# circuit goes straight to `qdk.qre`; the tracer reports the rotation and measurement counts as
compressed `repeat` blocks, so no logical-count formula is needed.

The trace query expands fine-grained rotations into T gates (`PSSPC`) and schedules the logical operations
with lattice surgery (`LatticeSurgery`); `slow_down_factor` trades runtime for fewer magic-state factories.
The ISA query supplies the surface-code instruction (`ThreeAux`) and a generic distillation model
(`RoundBasedFactory`).

The sweep ranges are given explicitly rather than left at their defaults: with $\sim 10^{7}$ rotations in the
QPE ladder, each rotation must be synthesized far more accurately than the defaults allow, and the default
query returns no result that meets `max_error`.

In [ ]:
# this cell takes ~3 mins to run
from qdk.qre import LatticeSurgery, PSSPC, estimate, plot_estimates
from qdk.qre.models import Majorana, RoundBasedFactory, ThreeAux

MAJORANA_ERROR_RATE = 1e-5
ARCHITECTURE = Majorana(error_rate=MAJORANA_ERROR_RATE)
MAX_ERROR = 0.01                     # max allowed error of the estimate; 0.5 was a very loose 50% tolerance

def estimate_physical(circuit: Circuit, name: str):
    """Qubit/runtime Pareto frontier for the given circuit on the Majorana architecture.

    Tracer memory grows like 2**num_bits * num_divisions * n_terms, which is what
    the timeout guards against.
    """
    application = circuit.get_qre_application()
    trace_query = (
        application.q()
        * PSSPC.q(num_ts_per_rotation=list(range(20, 45, 2)))
        * LatticeSurgery.q(slow_down_factor=[1.0 * j for j in range(1, 20)])
    )
    isa_query = ThreeAux.q() * RoundBasedFactory.q(code_query=ThreeAux.q())
    return estimate(application, ARCHITECTURE, isa_query, trace_query, max_error=MAX_ERROR, name=name)

estimates = estimate_physical(circuit, f"{EXAMPLE_NUM_SITES}x{EXAMPLE_NUM_SITES} lattice")
print(estimates)

## Sweeping the lattice sizes

For each lattice we build the qubit Hamiltonian, derive $(m, r)$ from its target precision, compile the QPE
circuit and estimate it. The table reports the derived parameters together with the fastest point on each
Pareto frontier.

Here $m$ stays constant across the sweep and $r$ is pinned at its floor of 1: the precision target grows with
the lattice ($\epsilon \propto N$) at the same rate as $\lambda$, so $2\lambda/\epsilon$ is size-independent,
while the ideal step count $t_0\sqrt{W/\epsilon_{\mathrm{Trotter}}} \propto N^{-1}$ (both $t_0 \propto 1/N$ and,
under the root, $W, \epsilon_{\mathrm{Trotter}} \propto N$ cancelling) is already below 1 for every $L \ge 6$.
So each larger lattice needs only a single Strang step; only the degenerate $L = 4$ tile is rounded up to
$r = 2$ by its nonzero $W_{\mathrm{PLAQ}}$ bound.

In [ ]:
# This cell takes 2 hrs
tables = []
rows = []

for size in BENCHMARK_LATTICE_SIZES:
    operator = qubit_operator(size)
    parameters = qpe_parameters(operator, target_precision(size), size)
    initial_state = reference_state_prep(size * size, num_electrons(size))
    table = estimate_physical(qpe_circuit(operator, parameters, initial_state, size), f"{size}x{size}")
    tables.append(table)

    fastest = min(table, key=lambda entry: entry.runtime)
    rows.append({
        "L": size,
        "sites": size * size,
        "qubits": operator.num_qubits,
        "terms": len(operator.pauli_strings),
        "electrons": num_electrons(size),
        "lambda": parameters.one_norm,
        "W_PLAQ / N": parameters.error_constant / (size * size),
        "num_bits": parameters.num_bits,
        "num_divisions": parameters.num_divisions,
        "physical qubits": fastest.qubits,
        "runtime (h)": fastest.runtime / 3.6e12,
    })
    print(f"{size}x{size} lattice: {fastest.runtime / 3.6e12} h {fastest.qubits} physical qubits")

pd.DataFrame(rows).set_index("L")

In [ ]:
fig = plot_estimates(tables, runtime_unit="hours", figsize=(12, 7))
ax = fig.axes[0]
ax.set_title("2D Fermi-Hubbard ground-state QPE, Majorana architecture")
fig.set_layout_engine("constrained")
fig

## How far the sweep can go

Two independent ceilings bound the sweep, and neither is the resource estimation itself.

1. **The mapper cap.** The sparse on-demand Majorana mapper supports $131{,}072$ qubits. At $2L^2$ qubits
   this permits $L \le 256$; larger lattices cannot be mapped. The analytic parameter scan below derives its
   endpoint from this ceiling because it only evaluates closed-form quantities.
2. **Hamiltonian-build memory, which binds first.** Building the qubit Hamiltonian is the bottleneck. Peak RSS
   for the build alone (one fresh process per size, measured here on this branch) is $2.0$ GB at $L = 60$,
   $5.6$ GB at $L = 80$, $13.7$ GB at $L = 100$, and $24.0$ GB at $L = 120$; the growth is roughly $L^{3.7}$,
   putting $L = 200$ near $220$ GB by extrapolation. The executed sweep now reaches `MAX_SWEEP_SIZE = 200`,
   so run it only on a node with sufficient RAM or lower that setting.
3. **Tracer memory**, which grows like $2^{m} \times r \times n_{\mathrm{terms}}$. Two choices keep this in
   check: iterative QPE compiles a *single* controlled unitary instead of the whole $m$-rung ladder, and the
   budget planner above picks the $m$ that minimizes $2^{m} r$. This is also why the loose
   `trotter_steps_naive` bound is unusable here: ignoring commutation it returns a size-independent step
   count (since $\lambda \propto N$ cancels against $t_0 = \pi/\lambda$) many times larger than the commutator
   bound, inflating the tracer's $2^{m} \times r \times n_{\mathrm{terms}}$ memory far beyond available RAM
   even at small $L$.

Qubitization-based alternatives such as FOQCS or SOSSA would replace the product formula in
`hamiltonian_unitary_builder` and are worth revisiting for the same benchmark:

- [2601.18767v1] Practical block encodings of matrix polynomials that can also be trivially controlled
- [2602.05069v1] Near-frustration-free electronic structure Hamiltonian representations and lower bound certificates

In [ ]:
import matplotlib.pyplot as plt

# For a periodic square lattice, there are 2N edges and two spin sectors.
# The hopping terms contribute 4|T|N to lambda and the on-site terms |U|N.
# Plaquette tiling needs even sides of at least four, bar the degenerate 2x2 torus. This scan is closed-form (no
# Hamiltonian is built), so derive its endpoint from the sparse Majorana mapper's qubit ceiling.
MAX_MAPPER_LATTICE_SIZE = math.isqrt(MAPPER_QUBIT_LIMIT // 2)
scaling_sizes = np.arange(4, MAX_MAPPER_LATTICE_SIZE + 1, 2)
scaling_sites = scaling_sizes**2
scaling_one_norm = (4 * abs(HOPPING_T) + abs(COULOMB_U)) * scaling_sites
scaling_precision = TARGET_PRECISION_PER_SITE * scaling_sites

# Reuse the planner so the analytic sweep and the executed lattices cannot drift apart.
scaling_plans = [
    plan_qpe(
        one_norm,
        precision,
        PlaquetteTrotter._plaquette_error_constant(int(size), int(size), HOPPING_T, COULOMB_U),
    )
    for one_norm, precision, size in zip(scaling_one_norm, scaling_precision, scaling_sizes)
]

parameter_scaling = pd.DataFrame(
    {
        "one_norm": scaling_one_norm,
        "evolution_time": [plan.evolution_time for plan in scaling_plans],
        "num_bits": [plan.num_bits for plan in scaling_plans],
        "num_divisions": [plan.num_divisions for plan in scaling_plans],
    },
    index=pd.Index(scaling_sizes, name="L"),
)

# Keep the analytic sweep tied to the explicitly constructed example.
assert np.isclose(parameter_scaling.loc[EXAMPLE_NUM_SITES, "one_norm"], parameters.one_norm)
assert np.isclose(parameter_scaling.loc[EXAMPLE_NUM_SITES, "evolution_time"], parameters.evolution_time)
assert parameter_scaling.loc[EXAMPLE_NUM_SITES, "num_bits"] == parameters.num_bits

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
series = [
    ("one_norm", r"Coefficient 1-norm $\lambda$", "log"),
    ("evolution_time", r"Evolution time $t_0$", "log"),
    ("num_bits", "QPE phase bits", "linear"),
]
for ax, (column, title, yscale) in zip(axes, series):
    ax.plot(parameter_scaling.index, parameter_scaling[column], linewidth=2)
    value = parameter_scaling.loc[EXAMPLE_NUM_SITES, column]
    ax.scatter([EXAMPLE_NUM_SITES], [value], color="tab:red", zorder=3)
    ax.annotate(
        f"L={EXAMPLE_NUM_SITES}\n{value:.6g}",
        (EXAMPLE_NUM_SITES, value),
        xytext=(8, 8),
        textcoords="offset points",
    )
    ax.set(title=title, xlabel="Linear lattice size L", yscale=yscale)
    ax.grid(True, which="both", alpha=0.3)

fig.suptitle("2D Fermi-Hubbard QPE parameter scaling")
fig.set_layout_engine("constrained")
fig